# AIRF 4201 · Module 9 — Revenue Management Challenge
## You are the airline revenue-management team

**Goal:** Select how many early Basic bookings to accept, how many seats to leave available for late Flex requests, and whether to overbook. Test a strategy, examine the financial and service consequences, revise, and defend your recommendation.

**Time:** ~12 minutes in teams. **No calculators required.**

**How to run:** Click **Runtime → Run all** in Google Colab. Wait for the controls to appear below the last cell.

**Decision rule:** You may not choose a policy that could deny boarding to **more than 3 people** in the modeled worst case. If your strategy violates this rule, it is not feasible.

This is a simplified **teaching simulation**, not operational booking advice. It assumes Basic customers are the first denied boarding, no-show counts apply to Basic passengers, all Flex passengers show up, denied passengers do not earn ticket or ancillary revenue, and denied boarding has an additional $900 cost.

### Case information (given to all teams)

| Input | Value |
|---|---:|
| Aircraft capacity | 180 seats |
| Basic fare (early demand 170) | $130 |
| Flex fare (late demand uncertain) | $290 |
| Flex high demand | 50 requests, probability 60% |
| Flex low demand | 20 requests, probability 40% |
| Basic ancillary revenue per flown passenger | $24 |
| Flex ancillary revenue per flown passenger | $8 |
| Basic no-shows | 4 (25%), 7 (50%), or 10 (25%) |
| Additional cost per involuntary denial | $900 |
| Worst-case denied boarding policy limit | 3 |

**Your decisions:** Basic booking limit and overbooking authorization. The simulator infers late Flex capacity = **180 + overbooking allowance − Basic booking limit**.

**Before calculating:** Estimate mentally: Does raising overbooking from 0 to 6 help revenue? What service risk might it introduce?

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
from math import isclose

CAPACITY = 180
BASIC_DEMAND = 170
FLEX_DEMAND = [(50, 0.60), (20, 0.40)]
BASIC_NO_SHOWS = [(4, 0.25), (7, 0.50), (10, 0.25)]
DENIAL_COST = 900
SERVICE_LIMIT = 3

def evaluate(basic_limit, overbook):
    # A basic booking limit is an accepted-booking CAP, not demand.
    basic = min(int(basic_limit), BASIC_DEMAND)
    total_sale_limit = CAPACITY + int(overbook)
    results = []
    for flex_requests, p_flex in FLEX_DEMAND:
        flex = min(flex_requests, max(0, total_sale_limit - basic))
        for no_shows, p_no in BASIC_NO_SHOWS:
            denied = max(0, basic + flex - no_shows - CAPACITY)
            # Case assumption: displaced/denied passengers come from Basic.
            basic_flown = max(0, basic - no_shows - denied)
            flex_flown = flex
            # A denied passenger's fare is refunded/not earned, plus penalty.
            revenue = basic_flown * (130 + 24) + flex_flown * (290 + 8) - DENIAL_COST * denied
            results.append({
                "flex_demand": flex_requests, "no_shows": no_shows,
                "prob": p_flex * p_no, "basic_sold": basic, "flex_sold": flex,
                "denied": denied, "flown": basic_flown + flex_flown,
                "revenue": revenue
            })
    expected = lambda key: sum(s["prob"] * s[key] for s in results)
    return {
        "expected_revenue": expected("revenue"),
        "expected_denied": expected("denied"),
        "chance_any_denied": sum(s["prob"] for s in results if s["denied"] > 0),
        "worst_denied": max(s["denied"] for s in results),
        "expected_flown": expected("flown"),
        "expected_basic_sold": expected("basic_sold"),
        "expected_flex_sold": expected("flex_sold"),
        "feasible": max(s["denied"] for s in results) <= SERVICE_LIMIT,
        "scenarios": results
    }

records = []
print("Simulator loaded. Use the team controls in the next cell.")

In [ ]:
team = widgets.Text(value="Team 1", description="Team:", style={"description_width":"90px"})
basic_limit = widgets.IntSlider(value=130, min=110, max=170, step=1, description="Basic limit:", continuous_update=False, style={"description_width":"90px"}, layout=widgets.Layout(width="550px"))
overbook = widgets.IntSlider(value=0, min=0, max=15, step=1, description="Overbook by:", continuous_update=False, style={"description_width":"90px"}, layout=widgets.Layout(width="550px"))
calculate = widgets.Button(description="Calculate outcome", button_style="primary", icon="calculator")
comparison = widgets.Button(description="Compare team attempts", icon="table")
reset = widgets.Button(description="Clear attempts", icon="trash")
output = widgets.Output()

def on_calculate(_):
    b, o = basic_limit.value, overbook.value
    result = evaluate(b, o)
    entries = {
        "Team": team.value.strip() or "Unnamed team", "Basic limit": b,
        "Overbook": o, "Flex seats held": max(0, CAPACITY + o - b),
        "Expected revenue": result["expected_revenue"],
        "P(denial)": result["chance_any_denied"],
        "Worst denials": result["worst_denied"],
        "Expected denials": result["expected_denied"],
        "Policy": "PASS" if result["feasible"] else "FAIL"
    }
    records.append(entries)
    with output:
        clear_output()
        print(f"TEAM DECISION: {entries['Team']}")
        print(f"Basic cap: {b} | Flex booking capacity: {entries['Flex seats held']} | Overbook allowance: {o}")
        print(f"Expected revenue: ${result['expected_revenue']:,.0f}")
        print(f"Probability of any denied boarding: {result['chance_any_denied']:.0%}")
        print(f"Expected passengers denied per flight: {result['expected_denied']:.2f}")
        print(f"Worst-case passengers denied: {result['worst_denied']}")
        print(f"Service rule (maximum 3 denied): {entries['Policy']}")
        if not result["feasible"]:
            print("Your policy is NOT eligible for selection. Revise and run again.")
        print("\nDiscuss: What changed? Is your additional expected revenue worth the service risk?")

def on_compare(_):
    with output:
        clear_output()
        if not records:
            print("No team attempts yet. Calculate a strategy first.")
            return
        df = pd.DataFrame(records).copy()
        df["Expected revenue"] = df["Expected revenue"].map(lambda x: f"${x:,.0f}")
        df["P(denial)"] = df["P(denial)"].map(lambda x: f"{x:.0%}")
        df["Expected denials"] = df["Expected denials"].map(lambda x: f"{x:.2f}")
        display(df)

def on_reset(_):
    records.clear()
    with output:
        clear_output()
        print("Attempts cleared.")

calculate.on_click(on_calculate)
comparison.on_click(on_compare)
reset.on_click(on_reset)
display(widgets.VBox([
    widgets.HTML("<h3>Set your team's strategy</h3>"),
    team, basic_limit, overbook,
    widgets.HBox([calculate, comparison, reset]),
    output
]))

### Team challenge · 4 minutes

1. **Round 1:** Choose and calculate one strategy. You may test a second policy after discussing why it should improve results.
2. **Round 2:** Compare your results with another team. Identify a strategy that produces **higher expected revenue** and one that creates **less denied-boarding risk**.
3. **Defend:** Recommend **one feasible policy** using three numbers: expected revenue, probability of denial, worst-case denied passengers.

**Debate prompt:** If your strategy earns several hundred dollars more per flight but occasionally denies boarding, is the tradeoff acceptable? What if the next flight has no available seats for disrupted passengers?

### Instructor-led surprise scenario

After teams commit to a recommendation, the instructor announces: **The service rule tightens from 3 to 0 denied boardings in the worst case.** No need to re-run code: look at your reported worst-case denials. Which teams must change their decision, and what revenue do they sacrifice?

**Important:** Team attempts are saved **only within the currently running Colab session**. Different teams opening their own notebook do not automatically share a leaderboard. For a class comparison, project team-reported outcomes or run multiple team entries on one instructor notebook.